# LABORATORIO N. 08

## Semana 8: Machine Learning No Supervisado - Clustering con K-Means en Google Colab

**Curso:** Fundamentos de Gestion de Datos  
**Tema:** Segmentacion de clientes usando Google Colab, Kaggle y SQLite  
**Dataset:** Mall Customer Segmentation Data, Kaggle  
**Fuente:** https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

## I. Elemento de la capacidad terminal

Aplica el algoritmo K-Means para segmentar clientes a partir de una base de datos SQLite. El flujo inicia con una base real descargada desde Kaggle, crea una tabla en SQLite, consulta los datos mediante SQL, determina el numero optimo de clusters, interpreta los segmentos y guarda los resultados en una nueva tabla de la base de datos.

## II. Resultado esperado

El codigo esta preparado para ejecutarse en **Google Colab**. Al finalizar el laboratorio, el estudiante habra construido un proceso completo:

1. Descargar un dataset desde Kaggle o subir el CSV manualmente a Colab.
2. Convertir el dataset en una base de datos SQLite.
3. Consultar la tabla con SQL.
4. Preparar variables numericas para clustering.
5. Aplicar K-Means.
6. Interpretar los segmentos de clientes.
7. Guardar el `cluster_id` en SQLite.

## III. Caso de negocio

Un centro comercial desea segmentar a sus clientes para disenar campanas diferenciadas. La informacion disponible incluye edad, genero, ingreso anual y puntaje de gasto. El objetivo es identificar grupos de clientes con comportamientos similares para orientar acciones de marketing.

In [ ]:
# Paso 1: instalar y cargar librerias

# En Google Colab, ejecuta esta celda para instalar kagglehub si no esta disponible.
try:
    import kagglehub
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## Paso 2: descargar o subir la base en Colab

Se usara el dataset **Mall Customer Segmentation Data**. Este dataset contiene 200 clientes y las variables `CustomerID`, `Gender`, `Age`, `Annual Income (k$)` y `Spending Score (1-100)`.

El laboratorio esta preparado para Google Colab con dos opciones:

- **Opcion A:** descargar automaticamente desde Kaggle usando `kagglehub`.
- **Opcion B:** si la descarga falla, subir manualmente el archivo `Mall_Customers.csv` con el boton de carga de Colab.

In [ ]:
# Paso 2: descargar dataset desde Kaggle o subirlo manualmente a Colab

DATASET_HANDLE = "vjchoudhary7/customer-segmentation-tutorial-in-python"

csv_path = None

try:
    dataset_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print("Dataset descargado desde Kaggle:", dataset_path)

    csv_files = list(dataset_path.glob("*.csv"))
    if csv_files:
        csv_path = csv_files[0]
except Exception as error:
    print("No se pudo descargar automaticamente desde Kaggle.")
    print("Motivo:", error)

if csv_path is None:
    print("Sube manualmente el archivo Mall_Customers.csv desde tu computadora.")
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No se subio ningun archivo.")
        csv_path = Path(next(iter(uploaded.keys())))
    except ImportError:
        raise RuntimeError(
            "Este bloque esta pensado para Google Colab. "
            "Si lo ejecutas localmente, coloca Mall_Customers.csv en la misma carpeta del notebook."
        )

print("Archivo CSV usado:", csv_path)

df_raw = pd.read_csv(csv_path)
print("Dimension original:", df_raw.shape)
display(df_raw.head())

print("\nInformacion del dataset:")
df_raw.info()

## Paso 3: crear una base de datos SQLite

Aunque el origen es Kaggle, el laboratorio debe trabajar con una base de datos. Por ello, el CSV se carga en SQLite y luego se vuelve a leer con una consulta SQL. Este paso convierte el ejercicio en un flujo de gestion de datos y no solo en analisis de un archivo plano.

In [ ]:
# Paso 3: crear base de datos SQLite desde el dataset de Kaggle

DB_PATH = Path("mall_customers_kaggle.db")
TABLE_CLIENTES = "clientes_mall"

conn = sqlite3.connect(DB_PATH)

# Guardar datos originales en SQLite
df_raw.to_sql(TABLE_CLIENTES, conn, if_exists="replace", index=False)

# Verificar tablas existentes
tablas = pd.read_sql_query(
    '''
    SELECT name AS tabla
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    ''',
    conn
)
display(tablas)

# Verificar estructura de la tabla
schema = pd.read_sql_query(f"PRAGMA table_info('{TABLE_CLIENTES}')", conn)
display(schema)

# Leer datos usando SQL
df = pd.read_sql_query(f"SELECT * FROM '{TABLE_CLIENTES}'", conn)
print(f"Datos leidos desde SQLite: {df.shape[0]} filas x {df.shape[1]} columnas")
display(df.head())

### P1

**Pregunta:** Que evidencia demuestra que el laboratorio trabaja con una base de datos?

**Respuesta sugerida:** El laboratorio crea una base SQLite llamada `mall_customers_kaggle.db`, guarda el dataset en la tabla `clientes_mall`, inspecciona el esquema con `PRAGMA table_info` y carga los datos con una consulta SQL. Por tanto, el modelo se alimenta desde una tabla de base de datos y no directamente desde el CSV.

## Paso 4: limpieza y seleccion de variables

Para K-Means se seleccionan variables numericas que describen el perfil del cliente. En este caso:

- `Age`: edad del cliente.
- `Annual Income (k$)`: ingreso anual en miles de dolares.
- `Spending Score (1-100)`: puntaje de gasto asignado por el centro comercial.

No se usa `CustomerID` porque es un identificador, no una caracteristica de comportamiento.

In [ ]:
# Paso 4: limpieza basica y seleccion de variables

print("Columnas disponibles:")
print(df.columns.tolist())

print("\nValores nulos por columna:")
display(df.isna().sum())

features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

X = df[features].copy()

print("Variables seleccionadas para clustering:", features)
display(X.describe().round(2))

# Grafica exploratoria inicial
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Gender",
    palette="Set2",
    s=60
)
plt.title("Relacion entre ingreso anual y puntaje de gasto")
plt.xlabel("Ingreso anual (k$)")
plt.ylabel("Puntaje de gasto (1-100)")
plt.tight_layout()
plt.show()

### P2

**Pregunta:** Por que se seleccionan estas variables para el clustering?

**Respuesta sugerida:** Se seleccionan `Age`, `Annual Income (k$)` y `Spending Score (1-100)` porque representan caracteristicas numericas del perfil del cliente. La edad describe el perfil demografico, el ingreso anual aproxima la capacidad economica y el puntaje de gasto refleja comportamiento de compra. En cambio, `CustomerID` se excluye porque solo identifica registros.

## Paso 5: estandarizacion

K-Means calcula distancias entre registros. Si las variables tienen escalas diferentes, la variable con mayor rango puede dominar el resultado. Por eso se aplica `StandardScaler`.

In [ ]:
# Paso 5: estandarizacion

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)

print("Estadisticas antes de estandarizar:")
display(X.describe().round(2))

print("Estadisticas despues de estandarizar:")
display(X_scaled_df.describe().round(2))

### P3

**Pregunta:** Que cambio despues de estandarizar y por que es importante?

**Respuesta sugerida:** Despues de estandarizar, las variables quedan con media cercana a 0 y desviacion estandar cercana a 1. Esto es importante porque K-Means usa distancias; si no se estandariza, el ingreso anual podria pesar mas que la edad o el puntaje de gasto solo por tener una escala numerica mayor.

## Paso 6: elegir el numero optimo de clusters

Se evaluan dos criterios:

- **Metodo del codo:** observa cuando la inercia deja de disminuir fuertemente.
- **Silhouette score:** mide que tan separados y compactos estan los clusters.

In [ ]:
# Paso 6: metodo del codo y silhouette

rango_k = range(2, 11)
inertias = []
silhouettes = []

for k in rango_k:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = modelo.fit_predict(X_scaled)
    inertias.append(modelo.inertia_)
    silhouettes.append(silhouette_score(X_scaled, etiquetas))

metricas = pd.DataFrame({
    "k": list(rango_k),
    "inercia": inertias,
    "silhouette": silhouettes
})

display(metricas.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(metricas["k"], metricas["inercia"], marker="o")
axes[0].set_title("Metodo del codo")
axes[0].set_xlabel("Numero de clusters (k)")
axes[0].set_ylabel("Inercia")
axes[0].set_xticks(list(rango_k))

axes[1].plot(metricas["k"], metricas["silhouette"], marker="o", color="green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Numero de clusters (k)")
axes[1].set_ylabel("Silhouette")
axes[1].set_xticks(list(rango_k))

plt.tight_layout()
plt.show()

# En este dataset es comun que k=5 sea interpretable para marketing:
# bajo ingreso/bajo gasto, bajo ingreso/alto gasto, medio, alto ingreso/bajo gasto y alto ingreso/alto gasto.
k_optimo = 5
print("k seleccionado:", k_optimo)

### P4

**Pregunta:** Por que se puede seleccionar k=5?

**Respuesta sugerida:** Se puede seleccionar k=5 porque el grafico del codo muestra una reduccion importante de la inercia hasta un punto donde la mejora posterior es menor. Ademas, desde el punto de vista de negocio, cinco segmentos permiten distinguir clientes de bajo ingreso/bajo gasto, bajo ingreso/alto gasto, comportamiento medio, alto ingreso/bajo gasto y alto ingreso/alto gasto. Esta separacion es util para disenar campanas diferenciadas.

## Paso 7: aplicar K-Means

Se entrena el modelo final y se agrega la columna `cluster_id` al dataset.

In [ ]:
# Paso 7: modelo final K-Means

kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df["cluster_id"] = kmeans.fit_predict(X_scaled)

distribucion = (
    df["cluster_id"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster_id")
    .reset_index(name="clientes")
)
distribucion["porcentaje"] = (distribucion["clientes"] / len(df) * 100).round(2)

display(distribucion)

# Centroides en escala original para interpretacion
centroides = scaler.inverse_transform(kmeans.cluster_centers_)
centroides_df = pd.DataFrame(centroides, columns=features)
centroides_df.insert(0, "cluster_id", range(k_optimo))

display(centroides_df.round(2))

## Paso 8: perfil de los segmentos

El perfil del cluster permite convertir los resultados tecnicos en informacion de negocio.

In [ ]:
# Paso 8: perfil estadistico por cluster

perfil = (
    df.groupby("cluster_id")[features]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

display(perfil)

# Nombres interpretativos basados en los centroides
def asignar_nombre(row):
    ingreso = row["Annual Income (k$)"]
    gasto = row["Spending Score (1-100)"]

    if ingreso >= 70 and gasto >= 60:
        return "Alto ingreso y alto gasto"
    if ingreso >= 70 and gasto < 45:
        return "Alto ingreso y bajo gasto"
    if ingreso < 45 and gasto >= 60:
        return "Bajo ingreso y alto gasto"
    if ingreso < 45 and gasto < 45:
        return "Bajo ingreso y bajo gasto"
    return "Cliente promedio"

centroides_df["segmento"] = centroides_df.apply(asignar_nombre, axis=1)
display(centroides_df.round(2))

df = df.merge(
    centroides_df[["cluster_id", "segmento"]],
    on="cluster_id",
    how="left"
)
display(df.head())

### P5

**Pregunta:** Describe los segmentos encontrados.

**Respuesta sugerida:** Los segmentos pueden interpretarse comparando ingreso anual y puntaje de gasto. Un segmento de alto ingreso y alto gasto representa clientes prioritarios para fidelizacion. Un segmento de alto ingreso y bajo gasto representa clientes con potencial de crecimiento. Un segmento de bajo ingreso y alto gasto puede responder bien a promociones. Un segmento de bajo ingreso y bajo gasto requiere acciones de bajo costo o campanas masivas. El segmento promedio puede mantenerse con comunicaciones regulares.

## Paso 9: visualizacion de clusters

In [ ]:
# Paso 9: visualizacion de clusters

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="segmento",
    palette="tab10",
    s=70
)
plt.title("Segmentacion de clientes con K-Means")
plt.xlabel("Ingreso anual (k$)")
plt.ylabel("Puntaje de gasto (1-100)")
plt.legend(title="Segmento", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df,
    x="segmento",
    y="Age"
)
plt.title("Distribucion de edad por segmento")
plt.xlabel("Segmento")
plt.ylabel("Edad")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## Paso 10: guardar resultados en SQLite

El resultado final se persiste en una tabla nueva para que pueda ser consultado con SQL.

In [ ]:
# Paso 10: guardar resultados en SQLite

TABLA_SEGMENTADA = "clientes_mall_segmentado"
TABLA_CENTROIDES = "clientes_mall_centroides"

df.to_sql(TABLA_SEGMENTADA, conn, if_exists="replace", index=False)
centroides_df.to_sql(TABLA_CENTROIDES, conn, if_exists="replace", index=False)

conn.commit()

verificacion = pd.read_sql_query(
    f'''
    SELECT cluster_id, segmento, COUNT(*) AS clientes
    FROM {TABLA_SEGMENTADA}
    GROUP BY cluster_id, segmento
    ORDER BY cluster_id;
    ''',
    conn
)

display(verificacion)

consulta_marketing = pd.read_sql_query(
    f'''
    SELECT segmento,
           ROUND(AVG(Age), 2) AS edad_promedio,
           ROUND(AVG("Annual Income (k$)"), 2) AS ingreso_promedio,
           ROUND(AVG("Spending Score (1-100)"), 2) AS gasto_promedio,
           COUNT(*) AS clientes
    FROM {TABLA_SEGMENTADA}
    GROUP BY segmento
    ORDER BY gasto_promedio DESC;
    ''',
    conn
)

display(consulta_marketing)

conn.close()
print("Base de datos actualizada:", DB_PATH.resolve())

### P6

**Pregunta:** Se guardo correctamente la segmentacion en SQLite?

**Respuesta sugerida:** Si. La segmentacion se guardo correctamente si la consulta SQL muestra la cantidad de clientes por `cluster_id` y `segmento` desde la tabla `clientes_mall_segmentado`. Esto es importante porque permite reutilizar los clusters en reportes, dashboards o campanas sin volver a ejecutar todo el modelo.

## Conclusion ejecutiva

El analisis permitio segmentar a los clientes del centro comercial en grupos diferenciados segun edad, ingreso anual y puntaje de gasto. Los segmentos de alto ingreso y alto gasto deben priorizarse con beneficios de fidelizacion; los de alto ingreso y bajo gasto requieren estrategias para aumentar su participacion; los de bajo ingreso y alto gasto pueden responder a promociones; y los clientes promedio pueden mantenerse con comunicaciones regulares. Al guardar los resultados en SQLite, la empresa puede consultar y actualizar la segmentacion desde una base de datos.